In [1]:
import os
import gc
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import anndata as ad
import scanpy as sc

%matplotlib inline

In [2]:
## Known Genes In Deleted Regions ##

GENES_22q11 = ["TBX1",  "COMT",  "DGCR8",    "CRKL",  "PRODH",  "HIRA",  "LZTR1",    "RTN4R",   "SNAP29",    "ZDHHC8"]

GENES_22q11_EXTENDED = ["RANBP1", "MED15",  "PI4KA",  "SLC25A1", "SEPT5", "GP1BB", "ZNF74",   
                     "CLDN5",    "ARVCF", "FAM230"]

GENES_3q29 = ["DLG1", "PAK2", "FBXO45", "RNF168", "TFRC", "BDH1", "PCYT1A", "ATP13A3"]

GENES_3q29_EXTENDED = ["WDR53", "CEP19",  "TM4SF1",  "SENP5", "MFI2", "UBXN7"]

GENES_15q13 = ["CHRNA7", "OTUD7A", "KLF13", "TRPM1", "FAN1", "MTMR10", "ARHGAP11B"]

GENES_15q13_EXTNEDED = ["POLR2M", "GOLGA8A", "GOLGA8B", "GOLGA8C", "GOLGA8D", "GOLGA8E", "GOLGA8F", "GOLGA8G", "GOLGA8H", "GOLGA8I"]

In [3]:
## Read in DEG Lists ##

CSV_PATH = '/mnt/sdb/scz_meta_analysis_processed/dge_signatures/dreamlet_dges/disease_analyses/'

dge_22q11 = pd.read_csv(Path(CSV_PATH) / 'dreamlet_22q11_dge_scz_results.csv')
dge_NRXN1 = pd.read_csv(Path(CSV_PATH) / 'dreamlet_NRNXN1_dge_scz_results.csv')
dge_3q29 = pd.read_csv(Path(CSV_PATH) / 'dreamlet_3q29_dge_scz_results.csv')
dge_15q13 = pd.read_csv(Path(CSV_PATH) / 'dreamlet_15q13_dge_scz_results.csv')
dge_Idiopathic = pd.read_csv(Path(CSV_PATH) / 'dreamlet_Idiopathic_dge_scz_results.csv')

In [4]:
UNIQUE_22q11 = np.unique(dge_22q11[(dge_22q11['adj.P.Val'] < 0.05)]['ID'])

UNIQUE_NRXN1 = np.unique(dge_NRXN1[(dge_NRXN1['adj.P.Val'] < 0.05)]['ID'])

UNIQUE_3q29 = np.unique(dge_3q29[(dge_3q29['adj.P.Val'] < 0.05)]['ID'])

UNIQUE_15q13 = np.unique(dge_15q13[(dge_15q13['adj.P.Val'] < 0.05)]['ID'])

UNIQUE_Idiopathic = np.unique(dge_Idiopathic[(dge_Idiopathic['adj.P.Val'] < 0.05)]['ID'])

In [5]:
## Pairwise Combinations

gene_sets = {"22q11": set(UNIQUE_22q11), "NRXN1": set(UNIQUE_NRXN1), 
             "3q29": set(UNIQUE_3q29),    "15q13": set(UNIQUE_15q13),
             "Idiopathic": set(UNIQUE_Idiopathic),}

for (name1, set1), (name2, set2) in combinations(gene_sets.items(), 2):
    overlap = set1 & set2
    print(f"{name1} ∩ {name2}: {len(overlap)} genes")
    if len(overlap) > 0:
        print(overlap)
        print()

22q11 ∩ NRXN1: 0 genes
22q11 ∩ 3q29: 0 genes
22q11 ∩ 15q13: 0 genes
22q11 ∩ Idiopathic: 2 genes
{'PHC1', 'SSC4D'}

NRXN1 ∩ 3q29: 0 genes
NRXN1 ∩ 15q13: 0 genes
NRXN1 ∩ Idiopathic: 0 genes
3q29 ∩ 15q13: 0 genes
3q29 ∩ Idiopathic: 2 genes
{'DYNLT2B', 'FBXO45'}

15q13 ∩ Idiopathic: 0 genes


In [6]:
## What percentage of the Known Genes in the Chr Block are DE ##

## make a heatmap for this ##

## 22q11 ##

print('22q11')
print(np.unique(dge_22q11[dge_22q11['adj.P.Val'] < 0.05]['ID'][dge_22q11[dge_22q11['adj.P.Val'] < 0.05]['ID'].isin(GENES_22q11)]))
print('22q11 extended')
print(np.unique(dge_22q11[dge_22q11['adj.P.Val'] < 0.05]['ID'][dge_22q11[dge_22q11['adj.P.Val'] < 0.05]['ID'].isin(GENES_22q11_EXTENDED)]))
print('\n')

## 3q29 ##

print('3q29')
print(np.unique(dge_3q29[dge_3q29['adj.P.Val'] < 0.05]['ID'][dge_3q29[dge_3q29['adj.P.Val'] < 0.05]['ID'].isin(GENES_3q29)]))
print('3q29 extended')
print(np.unique(dge_3q29[dge_3q29['adj.P.Val'] < 0.05]['ID'][dge_3q29[dge_3q29['adj.P.Val'] < 0.05]['ID'].isin(GENES_3q29_EXTENDED)]))
print('\n')

## NRXN1 ##

print('NRXN1')
print(np.unique(dge_NRXN1[dge_NRXN1['adj.P.Val'] < 0.05]['ID'][dge_NRXN1[dge_NRXN1['adj.P.Val'] < 0.05]['ID'].isin(['NRXN1'])]))
print('\n')

## 15q13 ##

print('15q13')
print(np.unique(dge_15q13[dge_15q13['adj.P.Val'] < 0.05]['ID'][dge_15q13[dge_15q13['adj.P.Val'] < 0.05]['ID'].isin([GENES_15q13])]))
print('15q13 extended')
print(np.unique(dge_15q13[dge_15q13['adj.P.Val'] < 0.05]['ID'][dge_15q13[dge_15q13['adj.P.Val'] < 0.05]['ID'].isin([GENES_15q13_EXTNEDED])]))
print('\n')

## Idiopathic ##

print('Idiopathic')
print(np.unique(dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]['ID'][dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]['ID'].isin(GENES_22q11)]))
print(np.unique(dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]['ID'][dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]['ID'].isin(GENES_22q11_EXTENDED)]))
print(np.unique(dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]['ID'][dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]['ID'].isin(GENES_3q29)]))
print(np.unique(dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]['ID'][dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]['ID'].isin(GENES_3q29_EXTENDED)]))
print(np.unique(dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]['ID'][dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]['ID'].isin(GENES_15q13)]))
print(np.unique(dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]['ID'][dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]['ID'].isin(GENES_15q13_EXTNEDED)]))
print(np.unique(dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]['ID'][dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]['ID'].isin(['NRXN1'])]))

## Idiopathic SCZ Shares 2 genes in Common with 3q29 deletion and Nothing Else ##

22q11
['COMT' 'CRKL' 'HIRA' 'LZTR1' 'RTN4R' 'SNAP29' 'ZDHHC8']
22q11 extended
['ARVCF' 'MED15' 'PI4KA' 'RANBP1' 'SLC25A1']


3q29
['DLG1' 'FBXO45' 'PCYT1A' 'RNF168']
3q29 extended
['WDR53']


NRXN1
[]


15q13
[]
15q13 extended
[]


Idiopathic
[]
[]
['FBXO45' 'TFRC']
[]
[]
[]
[]


In [7]:
dge_22q11[(dge_22q11['adj.P.Val'] < 0.05) & (np.abs(dge_22q11['logFC']) > 1)]

,assay,ID,logFC,AveExpr,t,P.Value,adj.P.Val,B,z.std
3,Mixed Neurons FGF12+GRIN2B+CAMK2B+,DGCR6L,-1.005780,5.543906,-7.139504,6.647040e-10,0.000016,3.078634,-6.174269
30,Mesenchymal-like cells VIM+VCAN+SPARC+,C6orf47,1.059411,3.883652,5.483152,5.061716e-07,0.001607,5.938861,5.023959
34,Radial Glia SOX2+VIM+FABP7+,HIRA,-1.042272,3.166057,-5.502756,1.474575e-06,0.004144,-0.418815,-4.814651
35,Mesenchymal-like cells VIM+VCAN+SPARC+,IFI27,1.987861,3.392458,5.275109,1.515710e-06,0.004144,2.179954,4.809153
41,Radial Glia SOX2+PAX6+FABP7+,SLC25A1,-1.006556,5.840882,-5.017550,2.999994e-06,0.007030,0.486380,-4.670820
44,Radial Glia SOX2+PAX6+FABP7+,COMT,-2.124549,4.210205,-4.908568,4.015905e-06,0.008783,-0.907957,-4.610557
45,Radial Glia SOX2+VIM+FABP7+,ARVCF,-1.029302,3.941597,-6.089659,4.626996e-06,0.009900,2.785095,-4.581026
49,Mesenchymal-like cells VIM+VCAN+SPARC+,PIK3CA,-1.522734,4.096901,-5.037840,9.245791e-06,0.017921,4.222712,-4.434100
54,Radial Glia SOX2+VIM+FABP7+,COMT,-2.063470,4.120611,-4.594966,1.611281e-05,0.028833,-3.930001,-4.312898
56,Proliferative Radial Glia SOX2+HES6+TOP2A+,COMT,-2.073465,3.273161,-4.558106,1.692035e-05,0.029215,-2.852614,-4.302078


In [8]:
dge_22q11[(dge_22q11['adj.P.Val'] < 0.05) & (dge_22q11['assay'] == 'Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+')]

,assay,ID,logFC,AveExpr,t,P.Value,adj.P.Val,B,z.std
0,Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+,PI4KA,-0.820498,4.971860,-7.643632,3.560559e-11,0.000004,10.374025,-6.621299
1,Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+,MED15,-0.821131,4.531648,-7.388265,1.384501e-10,0.000007,6.960852,-6.417588
12,Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+,SNAP29,-0.772316,5.014354,-6.811021,2.954268e-08,0.000224,6.188181,-5.544126
14,Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+,CRKL,-0.796497,4.724569,-5.997730,4.843240e-08,0.000318,5.201524,-5.456972
15,Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+,C22orf39,-0.727337,5.706293,-5.954626,6.794210e-08,0.000418,3.263112,-5.396530
20,Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+,ESS2,-0.741827,4.140018,-5.784844,1.609722e-07,0.000754,4.854414,-5.239546
27,Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+,RANBP1,-0.634556,7.820246,-5.553408,4.388332e-07,0.001542,2.782138,-5.051291
32,Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+,ARVCF,-0.899626,3.588668,-5.340270,7.006262e-07,0.002090,3.825270,-4.961190
38,Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+,DGCR2,-0.861911,3.808937,-5.091005,2.016560e-06,0.004994,2.892076,-4.751758
46,Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+,RTN4R,-0.856013,4.226957,-4.904799,5.077799e-06,0.010633,2.310817,-4.561547


In [9]:
dge_22q11[dge_22q11['adj.P.Val'] < 0.05].groupby('assay').count()

,ID,logFC,AveExpr,t,P.Value,adj.P.Val,B,z.std
assay,,,,,,,,
Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+,12,12,12,12,12,12,12,12
Hindbrain Neurons NR2F2+PBX3+LHX1+,15,15,15,15,15,15,15,15
Mesenchymal-like cells VIM+VCAN+SPARC+,6,6,6,6,6,6,6,6
Mixed Neurons FGF12+GRIN2B+CAMK2B+,10,10,10,10,10,10,10,10
Proliferative Radial Glia SOX2+HES6+TOP2A+,10,10,10,10,10,10,10,10
Radial Glia SOX2+PAX6+FABP7+,9,9,9,9,9,9,9,9
Radial Glia SOX2+VIM+FABP7+,11,11,11,11,11,11,11,11


In [10]:
dge_NRXN1[dge_NRXN1['adj.P.Val'] < 0.05].groupby('assay').count()

,ID,logFC,AveExpr,t,P.Value,adj.P.Val,B,z.std
assay,,,,,,,,


In [11]:
dge_3q29[dge_3q29['adj.P.Val'] < 0.05].groupby('assay').count()

,ID,logFC,AveExpr,t,P.Value,adj.P.Val,B,z.std
assay,,,,,,,,
Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+,7,7,7,7,7,7,7,7
GABAergic GAD1+GAD2+CALB2+,5,5,5,5,5,5,5,5
Hindbrain Neurons NR2F2+PBX3+LHX1+,7,7,7,7,7,7,7,7
Mixed Neurons FGF12+GRIN2B+CAMK2B+,7,7,7,7,7,7,7,7
Proliferative Radial Glia SOX2+HES6+TOP2A+,8,8,8,8,8,8,8,8
Radial Glia SOX2+PAX6+FABP7+,6,6,6,6,6,6,6,6
Radial Glia SOX2+VIM+FABP7+,11,11,11,11,11,11,11,11


In [13]:
dge_15q13[dge_15q13['adj.P.Val'] < 0.05].groupby('assay').count()

,ID,logFC,AveExpr,t,P.Value,adj.P.Val,B,z.std
assay,,,,,,,,
Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+,3,3,3,3,3,3,3,3
GABAergic GAD1+GAD2+CALB2+,1,1,1,1,1,1,1,1
Hindbrain Neurons NR2F2+PBX3+LHX1+,1,1,1,1,1,1,1,1
Mixed Neurons FGF12+GRIN2B+CAMK2B+,1,1,1,1,1,1,1,1


In [14]:
dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05].groupby('assay').count()

,ID,logFC,AveExpr,t,P.Value,adj.P.Val,B,z.std
assay,,,,,,,,
Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+,441,441,441,441,441,441,441,441
GABAergic GAD1+GAD2+CALB2+,56,56,56,56,56,56,56,56
Hindbrain Neurons NR2F2+PBX3+LHX1+,121,121,121,121,121,121,121,121
Mesenchymal-like cells VIM+VCAN+SPARC+,147,147,147,147,147,147,147,147
Mixed Neurons FGF12+GRIN2B+CAMK2B+,34,34,34,34,34,34,34,34
Proliferative Radial Glia SOX2+HES6+TOP2A+,519,519,519,519,519,519,519,519
Radial Glia SOX2+PAX6+FABP7+,159,159,159,159,159,159,159,159
Radial Glia SOX2+VIM+FABP7+,327,327,327,327,327,327,327,327


In [15]:
## Focusing on Idiopathic SCZ ##

## To what degree are DEGs conserved across celltype ##

dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]

,assay,ID,logFC,AveExpr,t,P.Value,adj.P.Val,B,z.std
0,Proliferative Radial Glia SOX2+HES6+TOP2A+,ALAD,3.052905,2.824755,16.271148,6.344177e-21,6.243812e-16,53.005946,9.384124
1,Proliferative Radial Glia SOX2+HES6+TOP2A+,BLVRB,4.073698,5.689543,10.447764,2.266840e-16,1.115489e-11,23.195661,8.207053
2,Radial Glia SOX2+VIM+FABP7+,CERS4,3.180029,4.737593,9.621767,3.727709e-15,1.222912e-10,24.164181,7.863760
3,Proliferative Radial Glia SOX2+HES6+TOP2A+,ELOVL1,2.858966,3.220990,9.258769,4.689903e-13,1.153927e-08,18.202019,7.233995
4,Radial Glia SOX2+VIM+FABP7+,ZNF165,2.641726,1.035159,7.982792,3.805093e-12,7.489793e-08,17.282845,6.944238
...,...,...,...,...,...,...,...,...,...
1799,Proliferative Radial Glia SOX2+HES6+TOP2A+,HTATSF1,-0.823571,6.902916,-3.501434,9.113152e-04,4.980459e-02,-1.614194,-3.316565
1800,Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+,GINM1,-1.534939,5.492069,-3.484913,9.113990e-04,4.980459e-02,-0.637860,-3.316539
1801,Radial Glia SOX2+PAX6+FABP7+,JAKMIP2,-1.782013,5.269445,-3.554382,9.125631e-04,4.984053e-02,-0.655860,-3.316182
1802,Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+,GON4L,-1.527373,6.045581,-3.581194,9.148642e-04,4.993850e-02,-0.493868,-3.315478


In [16]:
## negative overlap and shared lists ##

## code to test difference of differences ##

In [22]:
print(dge_22q11[(dge_22q11['adj.P.Val'] < 0.05) & (dge_22q11['logFC'] > 1)]['ID'].shape)

print(dge_NRXN1[(dge_NRXN1['adj.P.Val'] < 0.05) & (dge_NRXN1['logFC'] > 1)]['ID'].shape)

print(dge_3q29[(dge_3q29['adj.P.Val'] < 0.05) & (dge_3q29['logFC'] > 1)]['ID'].shape)

print(dge_15q13[(dge_15q13['adj.P.Val'] < 0.05) & (dge_15q13['logFC'] > 1)]['ID'].shape)

print(dge_Idiopathic[(dge_Idiopathic['adj.P.Val'] < 0.05) & (dge_Idiopathic['logFC'] > 1)]['ID'].shape)

(3,)
(0,)
(6,)
(6,)
(1123,)


In [33]:
print(np.unique(dge_22q11[(dge_22q11['adj.P.Val'] < 0.05) & (dge_22q11['logFC'] > 1)]['ID']).tolist())

print(np.unique(dge_NRXN1[(dge_NRXN1['adj.P.Val'] < 0.05) & (dge_NRXN1['logFC'] > 1)]['ID'].tolist()))

print(np.unique(dge_3q29[(dge_3q29['adj.P.Val'] < 0.05) & (dge_3q29['logFC'] > 1)]['ID'].tolist()))

print(np.unique(dge_15q13[(dge_15q13['adj.P.Val'] < 0.05) & (dge_15q13['logFC'] > 1)]['ID'].tolist()))

print(np.unique(dge_Idiopathic[(dge_Idiopathic['adj.P.Val'] < 0.05) & (dge_Idiopathic['logFC'] > 1)]['ID']).tolist())

['C6orf47', 'IFI27', 'SSC4D']
[]
['CHCHD2' 'CTSF' 'ELAPOR2' 'NLRP2']
['BEX5' 'POTEI' 'TCEAL5']
['ABALON', 'ABCA7', 'ABCB10', 'ABCC10', 'ABCG4', 'ABCG8', 'ABHD14B', 'ABHD16A', 'ABHD4', 'ACAD10', 'ACADS', 'ACSL1', 'ADA', 'ADAM19', 'ADAM8', 'ADAMTS4', 'ADAMTSL4', 'ADAT2', 'ADCK1', 'ADCY7', 'ADCYAP1R1', 'ADIPOR1', 'AFAP1L2', 'AGAP4', 'AGAP6', 'AGER', 'AHCY', 'AKAP3', 'ALAD', 'ALDH16A1', 'ALDH2', 'AMACR', 'AMOT', 'ANGEL1', 'ANKDD1A', 'ANKFY1', 'ANKH', 'ANKLE1', 'ANKMY1', 'ANKRD24', 'ANKRD52', 'ANKRD9', 'ANLN', 'ANXA2R', 'ANXA4', 'AP4B1-AS1', 'AP5B1', 'APOBEC3C', 'APOC1', 'APOE', 'APOL2', 'ARFGAP1', 'ARHGEF11', 'ARHGEF17', 'ARHGEF39', 'ARHGEF40', 'ARHGEF7-AS2', 'ARID3A', 'ARL4D', 'ARL8B', 'ARMC7', 'ARMT1', 'ARNT', 'ARPC1B', 'ARRDC2', 'ASB1', 'ASH2L', 'ASIC4', 'ASNS', 'ATAD3C', 'ATF5', 'ATG14', 'ATG9A', 'ATL2', 'ATL3', 'ATOX1', 'ATP11A', 'ATP1B2', 'ATP5MGL', 'ATP6V1E1', 'ATP6V1FNB', 'ATRIP', 'ATXN7', 'AXIN1', 'AZU1', 'B3GNT2', 'B4GALT4', 'BAMBI', 'BBS9', 'BCL11B', 'BCL2L1', 'BCL2L11', 'BCL2L1